# Reseach Agent - Part II: Tool Use and Reflective Agents


By the end of this lab, you can:
- Chain steps into a research pipeline (**search → reflection → formatting**).
- Convert natural-language output into **styled HTML** suitable for sharing.

In [2]:
!pip install -r requirements.txt

In [3]:
# ================================
# Standard library imports
# ================================
import json

# ================================
# Third-party imports
# ================================
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, HTML

# ================================
# Local / project imports
# ================================
import research_tools

# ================================
# Environment setup
# ================================
load_dotenv()  # Load environment variables from .env file

# Instantiate OpenAI's client (you should use this in your graded functions)
CLIENT = OpenAI()

In [6]:
import unittests

ModuleNotFoundError: No module named 'unittests'

## Using Tools

You’ll use two research tools exposed in the `research_tools` module:
- **`arxiv_search_tool(query, max_results)`** – academic papers via arXiv API.
- **`tavily_search_tool(query, max_results, include_images)`** – general web search via Tavily.

Let's explore how the `arxiv_search_tool` works.

This tool searches arXiv and returns a list of papers with:
- `title`, `authors`, `published`, `summary`, `url`, and (if available) `link_pdf`.

Below, we run a quick test and print the results in a readable format. Next cell is editable so feel free to try some search queries:


In [7]:
# Test the arXiv search tool
topic = "linear algebra"

arxiv_results = research_tools.arxiv_search_tool(topic, max_results=3)

# Show formatted arxiv_results
for i, paper in enumerate(arxiv_results, 1):
    if "error" in paper:
        print(f"❌ Error: {paper['error']}")
    else:
        print(f"📄 Paper {i}")
        print(f"  Title     : {paper['title']}")
        print(f"  Authors   : {', '.join(paper['authors'])}")
        print(f"  Published : {paper['published']}")
        print(f"  URL       : {paper['url']}\n")


print("\n🧾 Raw arxiv_Results:\n")
print(json.dumps(arxiv_results, indent=2))

📄 Paper 1
  Title     : Linear Mappings of Free Algebra
  Authors   : Aleks Kleyn
  Published : 2010-03-08
  URL       : http://arxiv.org/abs/1003.1544v2

📄 Paper 2
  Title     : Non-linear positive maps between $C^*$-algebras
  Authors   : Ali Dadkhah, Mox Sal Moslehian
  Published : 2018-11-07
  URL       : http://arxiv.org/abs/1811.03128v1

📄 Paper 3
  Title     : Grüss type inequalities for positive linear maps on $C^*$-algebras
  Authors   : Ali Dadkhah, Mohammad Sal Moslehian
  Published : 2016-10-12
  URL       : http://arxiv.org/abs/1610.03868v1


🧾 Raw arxiv_Results:

[
  {
    "title": "Linear Mappings of Free Algebra",
    "authors": [
      "Aleks Kleyn"
    ],
    "published": "2010-03-08",
    "url": "http://arxiv.org/abs/1003.1544v2",
    "summary": "For arbitrary F-algebra, in which the operation of addition is defined, I explore biring of matrices of mappings. The sum of matrices is determined by the sum in F-algebra, and the product of matrices is determined by the pr

The `tavily_search_tool` calls the Tavily API to fetch web results. Returns a list of dicts:
- `title`, `content`, `url` (and optional image URLs when `include_images=True`).

Run the cell to inspect sample output. Next cell is editable so feel free to try some search queries:

In [8]:
# Test the Tavily search tool
topic = "retrieval-augmented generation applications"

tavily_results = research_tools.tavily_search_tool(topic)
for item in tavily_results:
    print(item)

{'title': 'What Is Retrieval-Augmented Generation aka RAG - NVIDIA Blog', 'content': "NVIDIA artist's concept of retrieval-augmented generation aka RAG. The court clerk of AI is a process called retrieval-augmented generation, or RAG for short. Retrieval-augmented generation is a technique for enhancing the accuracy and reliability of generative AI models with information fetched from specific and relevant data sources. The paper, with coauthors from the former Facebook AI Research (now Meta AI), University College London and New York University, called RAG “a general-purpose fine-tuning recipe” because it can be used by nearly any LLM to connect with practically any external resource. The NVIDIA AI Blueprint for RAG gives developers a foundational starting point for using NVIDIA NeMo Retriever models to build scalable, customizable data extraction and retrieval pipelines that deliver high accuracy and throughput. A significant part of this RAG pipeline is NVIDIA NeMo Retriever, which 

## Tool Mapping

In the next cell you will define a dictionary that maps tool names (strings) to the actual Python functions. This allows the model to call tools by name during tool-calling. This dictionary will be used in your first graded function:

In [9]:
# Tool mapping
TOOL_MAPPING = {
    "tavily_search_tool": research_tools.tavily_search_tool,
    "arxiv_search_tool": research_tools.arxiv_search_tool,
}

## Exercise 1: Generate Research Report with Tools
**Goal:** Implement `generate_research_report_with_tools(prompt)`.

In this exercise, you'll work on a function that generates a detailed research report with the assistance of online tools.

## Key Hints

### 1. Setting Up the Chat with the Language Model
- **Tool Selection**: Ensure that the tools are automatically selected by the model. Look into how to set `tool_choice` to "auto" within the function call. A helpful resource can be found in [OpenAI’s Function Calling Documentation](https://platform.openai.com/docs/guides/function-calling#tool-choice).
- **Parameter Configuration**: Consider the parameters already defined in your function, such as model, messages, and tools. Think about how these might be used in your setup.

### 2. Recording Tool Call Results
- **Understanding the `ChatCompletionMessage`** object will help you access the required attributes to save the messages. An example of `ChatCompletionMessage` looks like this:

```python
ChatCompletionMessage(
    content=None,
    refusal=None,
    role='assistant',
    annotations=[],
    audio=None,
    function_call=None,
    tool_calls=[
        ChatCompletionMessageFunctionToolCall(
            id='call_ymMki5TBB91efJhMPjgoqjop',
            function=Function(
                arguments='{"query":"radio observations of recurrent novae","max_results":5}',
                name='arxiv_search_tool'
            ),
            type='function'
        )
    ]
)
```
Assuming that `msg` if of type `ChatCompletionMessage`, if you wanted to get the `name` of a `tool_call` you can do something like:
```python
 for call in msg.tool_calls:
    tool_name = call.function.name
```
Finally, the `result` variable will be created by actually calling the function associated with each tool (`tool_func`).

By leveraging these hints, you'll work towards an implementation that enables robust data gathering and report generation through smart tool integration.

In [12]:
# Exercise 1
def generate_research_report_with_tools(prompt: str, model: str = "gpt-4o") -> str:
    """
    Generates a research report using OpenAI's tool-calling with arXiv and Tavily tools.

    Args:
        prompt (str): The user prompt.
        model (str): OpenAI model name.

    Returns:
        str: Final assistant research report text.
    """
    messages = [
        {
            "role": "system",
            "content": (
                "You are a research assistant that can search the web and arXiv to write detailed, "
                "accurate, and properly sourced research reports.\n\n"
                "🔍 Use tools when appropriate (e.g., to find scientific papers or web content).\n"
                "📚 Cite sources whenever relevant. Do NOT omit citations for brevity.\n"
                "🌐 When possible, include full URLs (arXiv links, web sources, etc.).\n"
                "✍️ Use an academic tone, organize output into clearly labeled sections, and include "
                "inline citations or footnotes as needed.\n"
                "🚫 Do not include placeholder text such as '(citation needed)' or '(citations omitted)'."
            )
        },
        {"role": "user", "content": prompt}
    ]

    # List of available tools
    tools = [research_tools.arxiv_tool_def, research_tools.tavily_tool_def]

    # Maximum number of turns
    max_turns = 10
    final_text = ""

    # Iterate for max_turns iterations
    for _ in range(max_turns):

        response = CLIENT.chat.completions.create(
            model=model,
            messages=messages,
            tools=tools,
            tool_choice="auto",
            temperature=1,
        )

        # Get the response from the LLM and append to messages
        msg = response.choices[0].message
        messages.append(msg)

        # Stop when the assistant returns a final answer (no tool calls)
        if not msg.tool_calls:
            final_text = msg.content
            print("✅ Final answer:")
            print(final_text)
            break

        # Execute tool calls and append results
        for call in msg.tool_calls:
            tool_name = call.function.name
            args = json.loads(call.function.arguments)
            print(f"🛠️ {tool_name}({args})")

            try:
                tool_func = TOOL_MAPPING[tool_name]
                result = tool_func(**args)
            except Exception as e:
                result = {"error": str(e)}

            # Keep track of tool use in a new message
            new_msg = {
                "role": "tool",
                "tool_call_id": call.id,
                "name": tool_name,
                "content": json.dumps(result)
            }

            # Append to messages
            messages.append(new_msg)

    return final_text

Run the following cell to check the correctness of your code. It might take a while so don't worry if it takes a couple of minutes to run:

In [13]:
# Test your code!
my_report = generate_research_report_with_tools(None)
my_report

BadRequestError: Error code: 400 - {'error': {'message': "Invalid value for 'content': expected a string, got null.", 'type': 'invalid_request_error', 'param': 'messages.[1].content', 'code': None}}

In [14]:
my_report = generate_research_report_with_tools("application of linear algebra in ML")
my_report

✅ Final answer:
Linear algebra is a fundamental component of machine learning (ML), underpinning many of the operations and algorithms used within the field. Below, I will outline its major applications in various aspects of ML:

### 1. Data Representation
In machine learning, datasets are often represented using matrices. Each row of a matrix can correspond to a data point, while each column can represent a feature of those data points. This matrix representation allows for efficient storage and manipulation of datasets using linear algebraic operations.

### 2. Model Representation
Linear models, such as linear regression, can be represented compactly using linear algebra. For instance, a linear model can be expressed in the form:
\[ \mathbf{y} = \mathbf{X}\mathbf{w} + \mathbf{b} \]
where:
- \(\mathbf{y}\) is the vector of responses,
- \(\mathbf{X}\) is the matrix of features,
- \(\mathbf{w}\) is the vector of weights,
- \(\mathbf{b}\) is the bias term.

### 3. Optimization
Many mach

'Linear algebra is a fundamental component of machine learning (ML), underpinning many of the operations and algorithms used within the field. Below, I will outline its major applications in various aspects of ML:\n\n### 1. Data Representation\nIn machine learning, datasets are often represented using matrices. Each row of a matrix can correspond to a data point, while each column can represent a feature of those data points. This matrix representation allows for efficient storage and manipulation of datasets using linear algebraic operations.\n\n### 2. Model Representation\nLinear models, such as linear regression, can be represented compactly using linear algebra. For instance, a linear model can be expressed in the form:\n\\[ \\mathbf{y} = \\mathbf{X}\\mathbf{w} + \\mathbf{b} \\]\nwhere:\n- \\(\\mathbf{y}\\) is the vector of responses,\n- \\(\\mathbf{X}\\) is the matrix of features,\n- \\(\\mathbf{w}\\) is the vector of weights,\n- \\(\\mathbf{b}\\) is the bias term.\n\n### 3. Optim

## Exercise 2: Reflection + Rewrite

**Goal:** Implement `reflection_and_rewrite(report)`.

In this task, your goal is to develop a function that takes a report, analyzes it, generates a structured reflection, and produces an improved version of the report.


## Key Steps

### Create a User Prompt

- **Objective**: Guide the language model to output a structured response in JSON format.
- **Format**: Ensure the output includes two keys, `"reflection"` and `"revised_report"`.
- **Details**: Your reflection should cover strengths, limitations, suggestions, and opportunities.

In [15]:
# Exercise 2
def reflection_and_rewrite(report, model: str = "gpt-4o-mini", temperature: float = 0.3) -> dict:
    """
    Generates a structured reflection AND a revised research report.
    Accepts raw text OR the messages list returned by generate_research_report_with_tools.

    Returns:
        dict with keys:
          - "reflection": structured reflection text
          - "revised_report": improved version of the input report
    """

    # Input can be plain text or a list of messages, this function detects and parses accordingly
    report = research_tools.parse_input(report)

    user_prompt = f"""
Analyze the following research report and respond ONLY with valid JSON in this exact format:

{{
  "reflection": "<brief structured reflection covering strengths, limitations, suggestions, and opportunities>",
  "revised_report": "<improved version of the report>"
}}

Important rules:
- Output ONLY valid JSON
- Do not include markdown fences
- Do not include any explanation outside JSON
- Keep the reflection concise but useful
- Improve clarity, structure, accuracy, and completeness in the revised report

Research report:
\"\"\"
{report}
\"\"\"
"""

    response = CLIENT.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are an academic reviewer and editor."},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature
    )

    # Extract output
    llm_output = response.choices[0].message.content.strip()

    # Check if output is valid JSON
    try:
        data = json.loads(llm_output)
    except json.JSONDecodeError:
        raise Exception("The output of the LLM was not valid JSON. Adjust your prompt.")

    return {
        "reflection": str(data.get("reflection", "")).strip(),
        "revised_report": str(data.get("revised_report", "")).strip(),
    }

In [ ]:
# Test your code!
my_reflection = reflection_and_rewrite(None)
my_reflection

In [16]:
#Solution
my_reflection = reflection_and_rewrite(my_report)
my_reflection

{'reflection': 'The report effectively outlines the applications of linear algebra in machine learning, demonstrating clarity and relevance. However, it lacks depth in some sections and could benefit from examples or references to specific algorithms. Suggestions include adding more detailed explanations of key concepts and improving transitions between sections. There is an opportunity to incorporate recent developments in ML that leverage linear algebra more explicitly.',
 'revised_report': 'Linear algebra is a fundamental component of machine learning (ML), underpinning many of the operations and algorithms used within the field. Below, I will outline its major applications in various aspects of ML:\n\n### 1. Data Representation\nIn machine learning, datasets are often represented using matrices. Each row of a matrix corresponds to a data point, while each column represents a feature of those data points. This matrix representation allows for efficient storage and manipulation of da

## Exercise 3: Convert Report to HTML
**Goal:** Implement `convert_report_to_html(report)`.
This exercise focuses on transforming a plain text research report into a well-structured HTML document.

## Key Steps

### 1. Create a User Prompt
- **Objective**: Instruct the model to transform plain text into HTML structure.
- **Format**: Ensure the output is valid, clean HTML with appropriate section headers, formatted paragraphs, and clickable links.
- **Details**: Preserve the citation style and request that the model responds only with HTML, without additional commentary.

### 2. Configure the Response Call
- **Parameters**: Use the specified model (e.g., `"gpt-4o"`) and set an appropriate temperature to balance creativity and accuracy.
- **Structure**: Configure the `CLIENT.chat.completions.create` call properly, using both system and user prompts to ensure a clear and focused task description.

In [17]:
# Exercise 3
def convert_report_to_html(report, model: str = "gpt-4o", temperature: float = 0.5) -> str:
    """
    Converts a plaintext research report into a styled HTML page using OpenAI.
    Accepts raw text OR the messages list from the tool-calling step.
    """

    # Input can be plain text or the messages list from the tool-calling step
    report = research_tools.parse_input(report)

    # System prompt is already provided
    system_prompt = "You convert plaintext reports into full clean HTML documents."

    # Build the user prompt instructing the model to return ONLY valid HTML
    user_prompt = f"""
Convert the following plaintext research report into a complete, clean HTML document.

Requirements:
- Return ONLY valid HTML
- Include <html>, <head>, and <body>
- Use semantic structure with headings, paragraphs, and lists where appropriate
- Preserve citations and URLs
- Make links clickable with <a href="...">
- Use simple embedded CSS for readability
- Do not include markdown
- Do not include explanations outside the HTML

Report:
\"\"\"
{report}
\"\"\"
"""

    # Call the LLM
    response = CLIENT.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature
    )

    # Extract the HTML from the assistant message
    html = response.choices[0].message.content.strip()

    return html

In [ ]:
# Test your code!
my_html = convert_report_to_html(None)
display(HTML(my_html))

In [18]:
# Test your code!
my_html = convert_report_to_html(my_reflection.get("revised_report"))
display(HTML(my_html))

### End-to-End Pipeline

Run this cell to execute the full workflow:

1. Generate a research report (tools).
2. Reflect on the report.
3. Convert the report to HTML.

> You should see the rendered HTML below and two concise reflections in the console.

In [19]:
# 1) Research with tools
prompt_ = "Radio observations of recurrent novae"
preliminary_report = generate_research_report_with_tools(prompt_)
print("=== Research Report (preliminary) ===\n")
print(preliminary_report)

# 2) Reflection on the report (use the final TEXT to avoid ambiguity)
reflection_text = reflection_and_rewrite(preliminary_report)   # <-- pass text, not messages
print("=== Reflection on Report ===\n")
print(reflection_text['reflection'], "\n")
print("=== Revised Report ===\n")
print(reflection_text['revised_report'], "\n")


# 3) Convert the report to HTML (use the TEXT and correct function name)
html = convert_report_to_html(reflection_text['revised_report'])

print("=== Generated HTML (preview) ===\n")
print((html or "")[:600], "\n... [truncated]\n")

# 4) Display full HTML
display(HTML(html))

🛠️ arxiv_search_tool({'query': 'radio observations of recurrent novae'})
✅ Final answer:
## Overview of Radio Observations of Recurrent Novae

The study of recurrent novae through radio observations is a pivotal aspect of understanding these stellar phenomena. Recurrent novae are characterized by repeated nova outbursts from the same star system, typically involving a white dwarf and a companion star.

### Notable Studies and Details

1. **Lesson Learned from (Some) Recurrent Novae**  
   - **Authors**: Elena Mason, Frederick M. Walters
   - **Publication Date**: March 12, 2013
   - **Summary**: This study discusses early decline and nebular spectra of the recurrent novae YY Dor and Nova LMC 2009, among others. These recurrent novae share similar spectral characteristics, suggesting a similar progenitor white dwarf and post-outburst phases. The analysis highlights common features that may point to consistent underlying mechanisms [arXiv link](http://arxiv.org/abs/1303.2776v1) [PDF](htt